# Inverse Design with a Tandem Neural Network

In this tutorial, we will use a **Mie-informed tandem network** to perform inverse design of a core-shell nanoparticle. Our goal is to design a particle that satisfies the **First Kerker condition**, which is characterized by zero backscattering ($a_1 = b_1$).

### Tandem Network Architecture
Instead of directly optimizing the geometric parameters, we will train a neural network (the "inverse network") that takes a **target scattering pattern** as input and predicts the optimal **geometrical parameters** (layer thicknesses) as output. 

During training, the predicted geometry is fed into the differentiable PyMieDiff solver (the "forward model") to compute the actual scattering pattern. The loss is then calculated as the Mean Squared Error (MSE) between the target and simulated patterns, and backpropagated through the physics solver to update the network weights.

### Problem Statement
**Objective:** Find a core-shell geometry (Silicon core, Silica shell) that exhibits zero backscattering at a specific wavelength.
**Target:** A highly directional scattering pattern where backscattering ($\theta = \pi$) is zero.
**Environment:** Air ($n = 1.0$).
**Wavelength:** Fixed at $\lambda = 650$ nm.

In [ ]:
# Installation check
try:
    import pymiediff
    print("pymiediff already installed.")
except ImportError:
    print("Installing pymiediff...")
    !pip install -q https://github.com/UoS-Integrated-Nanophotonics-group/MieDiff/archive/refs/heads/main.zip
    import pymiediff

# Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pymiediff as pmd

# Suppress warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Set default dtype to float64 for high precision physics
torch.set_default_dtype(torch.float64)

# Ensure consistent dark style across notebooks
plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#191919', 'axes.facecolor': '#191919', 'savefig.facecolor': '#191919'})

## 1. Define the Target Scattering Pattern

For the First Kerker condition, we want zero backscattering. We define a target angular scattering pattern over $\theta \in [0, 2\pi]$. A simple idealization of directional forward scattering is $I(\theta) \propto (1 + \cos(\theta))^2$, which vanishes exactly at $\theta = \pi$.

In [ ]:
num_angles = 100
theta = torch.linspace(0, 2 * torch.pi, num_angles)

# Idealized Kerker pattern: (1 + cos(theta))^2, normalized
target_pattern = (1 + torch.cos(theta))**2
target_pattern = target_pattern / torch.max(target_pattern)  # Normalize max to 1

fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(5, 5))
ax.plot(theta.numpy(), target_pattern.numpy(), color='cyan', linewidth=2)
ax.set_title("Target Scattering Pattern (Kerker Condition)", pad=20)
ax.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. Build the Tandem Neural Network

We design an inverse network (MLP) that takes the 100-point angular pattern as input and outputs 2 values representing the core and shell thicknesses. We use a Sigmoid activation at the output to bound the thicknesses within physically reasonable ranges (e.g., 10 to 150 nm).

In [ ]:
class InverseNetwork(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=64, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid()  # Outputs between 0 and 1
        )
        
        # Min and max thicknesses for core and shell in nm
        self.d_min = torch.tensor([2.5, 2.5])
        self.d_max = torch.tensor([500.0, 500.0])
        
    def forward(self, x):
        out = self.net(x)
        # Scale output to physical dimensions
        thicknesses = self.d_min + out * (self.d_max - self.d_min)
        return thicknesses

inverse_model = InverseNetwork(input_dim=num_angles, output_dim=2)

## 3. Define the Forward Pass

We define a function to compute the forward pass of the tandem network and the Mie solver, and compute the loss against the target pattern.

In [ ]:
def compute_forward_and_loss(target_pattern, inverse_model, k0, theta, eps_l, eps_env, criterion):
    # 1. Forward pass through Inverse Network
    d_pred = inverse_model(target_pattern)
    r_layers = torch.cumsum(d_pred, dim=-1)
    
    # 2. Forward pass through Physics Solver (PyMieDiff)
    simulated_pattern = pmd.multishell.angular_scattering(
        k0=k0,
        theta=theta,
        r_layers=r_layers,
        eps_layers=eps_l,
        eps_env=eps_env
    )
    
    # We use the unpolarized scattering intensity, normalized
    sim_unp = simulated_pattern["i_unpol"].squeeze()
    sim_unp_norm = sim_unp / torch.max(sim_unp)
    
    # 3. Compute Loss
    loss = criterion(sim_unp_norm, target_pattern)
    
    return loss, d_pred

## 4. The Optimization Loop

Here we set up the parameters, initialize the Adam optimizer, and run the standard PyTorch optimization loop.

In [ ]:
def initialize_optimization(inverse_model, wl_fixed=1550.0, lr=1e-4):
    optimizer = optim.Adam(inverse_model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    mat_core = pmd.materials.MatDatabase("si")
    mat_shell = pmd.materials.MatDatabase("SiO2")
    eps_env = torch.tensor([1.0])

    wl0 = torch.tensor([wl_fixed])
    k0 = (2 * torch.pi / wl0).reshape(1)
    e_core = mat_core.get_epsilon(wl0)
    e_shell = mat_shell.get_epsilon(wl0)
    eps_l = torch.stack([e_core, e_shell], dim=0).reshape(2, 1)

    history = {'loss': [], 'thicknesses': []}
    
    return optimizer, criterion, history, k0, eps_l, eps_env, wl0, mat_core, mat_shell

def log_metrics(step, num_iterations, loss, d_pred, history):
    history['loss'].append(loss.item())
    history['thicknesses'].append(d_pred.detach().numpy().copy())
    
    if (step + 1) % max(1, (num_iterations // 10)) == 0:
        print(f"Step {step+1:03d} | Loss: {loss.item():.6f} | r_core: {d_pred[0].item():.1f} nm, d_shell: {d_pred[1].item():.1f} nm")

In [ ]:
# Configurable optimization parameters
num_iterations = 2000
lr = 3.3e-3
wl_fixed = 1550.0

optimizer, criterion, history, k0, eps_l, eps_env, wl0, mat_core, mat_shell = initialize_optimization(inverse_model, wl_fixed, lr)

print("Starting tandem network optimization...\n")

# Clean Optimization Loop
for step in range(num_iterations):
    optimizer.zero_grad()                   # 1. Clear old gradients
    
    loss, d_pred = compute_forward_and_loss( # 2 & 3. Forward pass and Compute Loss
        target_pattern, inverse_model, 
        k0, theta, eps_l, eps_env, criterion
    )
    
    loss.backward()                         # 4. Backpropagation
    optimizer.step()                        # 5. Update weights
    
    log_metrics(step, num_iterations, loss, d_pred, history)  # 6. Log metrics

print("\nOptimization complete.")

# store final parameters
with torch.no_grad():
    final_d = inverse_model(target_pattern)
    r_final = torch.cumsum(final_d, dim=-1)

## 5. Results Visualization

### 5.1 Gradient Descent History
We plot the objective function (MSE Loss) and the evolution of the geometrical parameters (layer thicknesses) during optimization.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# panel 1: objective function
axes[0].semilogy(history['loss'], color='cyan', linewidth=2, label='Loss (MSE)')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss', color='cyan')
axes[0].tick_params(axis='y', labelcolor='cyan')
axes[0].set_title('Objective Function')
axes[0].grid(True, linestyle='--', alpha=0.3)

# panel 2: geometrical parameters
thicknesses_history = np.array(history['thicknesses'])
axes[1].plot(thicknesses_history[:, 0], label='Core Radius (r_core)')
axes[1].plot(thicknesses_history[:, 1], label='Shell Thickness (d_shell)')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Thickness (nm)')
axes[1].set_title('Geometrical Parameters')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Radiation Patterns
Comparison of the target and optimized scattering patterns.

#### 5.2.1 Compute radiation pattern

In [ ]:
with torch.no_grad():
    final_sim_pattern = pmd.multishell.angular_scattering(
        k0=k0,
        theta=theta,
        r_layers=r_final,
        eps_layers=eps_l,
        eps_env=eps_env
    )
    sim_unp_final = final_sim_pattern["i_unpol"].squeeze()
    sim_unp_final = sim_unp_final / torch.max(sim_unp_final)

#### 5.2.2 Plot radiation pattern

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(6, 6))
ax.plot(theta.numpy(), target_pattern.numpy(), color='white', linestyle='--', linewidth=2, label='Target Pattern')
ax.plot(theta.numpy(), sim_unp_final.numpy(), color='cyan', linewidth=2, label='Optimized Pattern')
ax.set_title("Radiation Pattern Comparison", pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True, linestyle='--', alpha=0.3)
plt.show()

### 5.3 Cross Sections & Dipole Magnitudes
We calculate the cross-sections over a wavelength range and inspect the magnitude of the electric ($a_1$) and magnetic ($b_1$) dipoles to verify the Kerker condition ($|a_1-b_1| \sim 0$ at $1550$ nm).

In [ ]:
wls = torch.linspace(400, 2000, 500)
ks = 2 * torch.pi / wls
eps_core_r = mat_core.get_epsilon(wls)
eps_shell_r = mat_shell.get_epsilon(wls)
eps_l_r = torch.stack([eps_core_r, eps_shell_r], dim=0)

with torch.no_grad():
    cs = pmd.multishell.cross_sections(
        k0=ks,
        r_layers=r_final,
        eps_layers=eps_l_r,
        eps_env=torch.tensor([1.0]),
        n_max=15
    )
    
    coeffs = pmd.multishell.mie_coefficients(
        k0=ks,
        r_layers=r_final,
        eps_layers=eps_l_r,
        eps_env=torch.tensor([1.0]),
        n_max=2
    )
    a1 = coeffs['a_n'][0]
    b1 = coeffs['b_n'][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cross-sections
axes[0].plot(wls.numpy(), cs["cs_ext"].squeeze().numpy(), label='Extinction', color='cyan')
axes[0].plot(wls.numpy(), cs["cs_sca"].squeeze().numpy(), label='Scattering', color='magenta')
axes[0].plot(wls.numpy(), cs["cs_abs"].squeeze().numpy(), label='Absorption', color='yellow')
axes[0].axvline(wl0.item(), color='white', linestyle='--', alpha=0.5, label=r'Design $\lambda$')
axes[0].legend()
axes[0].set_xlabel('Wavelength (nm)')
axes[0].set_ylabel('Cross section (nm$^2$)')
axes[0].set_title('Cross-Sections')
axes[0].grid(True, linestyle='--', alpha=0.3)

# Dipole magnitudes
axes[1].plot(wls.numpy(), torch.abs(a1-b1).squeeze().numpy(), label='$|a_1-b_1|$', color='cyan', linewidth=2)
# axes[1].plot(wls.numpy(), torch.abs(b1).squeeze().numpy(), label='$|b_1|$ (Magnetic Dipole)', color='magenta', linewidth=2)
axes[1].axvline(wl0.item(), color='white', linestyle='--', alpha=0.5, label=r'Design $\lambda$')
axes[1].legend()
axes[1].set_xlabel('Wavelength (nm)')
axes[1].set_ylabel('Magnitude')
axes[1].set_title('Dipole Magnitudes (Kerker Condition Check)')
axes[1].grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### 5.4 Local Fields
Visualization of the local field enhancements ($|E|^2$ and $|H|^2$) and the real parts of all Cartesian components.

In [ ]:
L = r_final[-1].item() + 50.0
res = 200

_x = torch.linspace(-L, L, res)
_z = torch.linspace(-L, L, res)
grid_x, grid_z = torch.meshgrid(_x, _z, indexing='ij')
probes = torch.stack([grid_x, torch.zeros_like(grid_x), grid_z], dim=-1).view(-1, 3)

with torch.no_grad():
    nf = pmd.multishell.nearfields(
        k0=k0,
        r_probe=probes,
        r_layers=r_final,
        eps_layers=eps_l,
        eps_env=torch.tensor([1.0]),
        n_max=20
    )

E_t = nf["E_t"].view(res, res, 3)
H_t = nf["H_t"].view(res, res, 3)

intensity_E = torch.sum(torch.abs(E_t)**2, dim=-1)
intensity_H = torch.sum(torch.abs(H_t)**2, dim=-1)

fig, axes = plt.subplots(2, 4, figsize=(20, 7))

# Electric fields (Top Row)
im0 = axes[0, 0].imshow(intensity_E.numpy().T, extent=[-L, L, -L, L], origin='lower', cmap='magma', vmax=np.percentile(intensity_E.numpy(), 99))
axes[0, 0].set_title('$|E|^2$')
fig.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(E_t[..., 0].real.numpy().T, extent=[-L, L, -L, L], origin='lower', cmap='bwr', vmin=-3, vmax=3)
axes[0, 1].set_title('Re($E_x$)')
fig.colorbar(im1, ax=axes[0, 1])

im2 = axes[0, 2].imshow(E_t[..., 1].real.numpy().T, extent=[-L, L, -L, L], origin='lower', cmap='bwr', vmin=-3, vmax=3)
axes[0, 2].set_title('Re($E_y$)')
fig.colorbar(im2, ax=axes[0, 2])

im3 = axes[0, 3].imshow(E_t[..., 2].real.numpy().T, extent=[-L, L, -L, L], origin='lower', cmap='bwr', vmin=-3, vmax=3)
axes[0, 3].set_title('Re($E_z$)')
fig.colorbar(im3, ax=axes[0, 3])

# Magnetic fields (Bottom Row)
im4 = axes[1, 0].imshow(intensity_H.numpy().T, extent=[-L, L, -L, L], origin='lower', cmap='magma', vmax=np.percentile(intensity_H.numpy(), 99))
axes[1, 0].set_title('$|H|^2$')
fig.colorbar(im4, ax=axes[1, 0])

im5 = axes[1, 1].imshow(H_t[..., 0].real.numpy().T, extent=[-L, L, -L, L], origin='lower', cmap='bwr', vmin=-3, vmax=3)
axes[1, 1].set_title('Re($H_x$)')
fig.colorbar(im5, ax=axes[1, 1])

im6 = axes[1, 2].imshow(H_t[..., 1].real.numpy().T, extent=[-L, L, -L, L], origin='lower', cmap='bwr', vmin=-3, vmax=3)
axes[1, 2].set_title('Re($H_y$)')
fig.colorbar(im6, ax=axes[1, 2])

im7 = axes[1, 3].imshow(H_t[..., 2].real.numpy().T, extent=[-L, L, -L, L], origin='lower', cmap='bwr', vmin=-3, vmax=3)
axes[1, 3].set_title('Re($H_z$)')
fig.colorbar(im7, ax=axes[1, 3])

for ax in axes.flatten():
    for r in r_final.numpy()[::-1]:
        ax.add_patch(plt.Circle((0,0), r, ec='w', fc='none', alpha=0.4))
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()